# 1. Download Solar.ChemDX data

Download the group archives available to your account, then optionally select
devices with a particular measurement. Each run creates a new folder and keeps
the downloaded ZIPs and full JSON records.

**Before starting:** download or clone the whole repository (including
`solarcell_tools.py`), install `requirements-notebooks.txt`, and open JupyterLab from the
repository folder. Use Python 3.10+. Generate an API key at
[Solar.ChemDX → Account](https://solar.chemdx.org/account).

Run the cells from top to bottom. You can enter credentials when prompted or set
`SOLARCELL_EMAIL` and `SOLARCELL_API_KEY` in your environment before starting Jupyter.
If your organisation uses a TLS proxy, set `REQUESTS_CA_BUNDLE` to its CA file.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from collections import Counter

from solarcell_tools import download_dataset, load_records, filter_records, measurement_types, write_records

DOWNLOAD_ROOT = Path("downloads")
GROUP_IDS = None  # All available groups, or a list such as [2, 3]. IDs are checked against the API.
MEASUREMENT = None  # All devices, or an exact key such as "SEM", "JV", "PL", "TRPL", "XRD".

## Authenticate and download

The API returns temporary ZIP URLs, normally valid for about five minutes. A
download receiving HTTP 403 refreshes its link once. Network timeouts, malformed
responses and authentication failures stop the run with a readable error.

No API key or signed URL is written to the output files. The `latest.json` pointer
is updated only after every selected group has downloaded and merged successfully.
Previous runs remain available if a new run fails.

In [ ]:
email = os.environ.get("SOLARCELL_EMAIL") or input("Solar.ChemDX account email: ").strip()
api_key = os.environ.get("SOLARCELL_API_KEY") or getpass("Solar.ChemDX API key: ")
try:
    records_file = download_dataset(email, api_key, DOWNLOAD_ROOT, GROUP_IDS)
finally:
    del api_key

## Inspect and optionally filter measurements

Filters run locally after download. Measurement names are matched exactly, ignoring
case: **PL does not include TRPL**. Both `analysis` and `analysisInfo` are checked
for non-empty entries. Metadata may indicate an attachment even when parsed
measurement values are absent. The API ZIPs contain device JSON; this notebook
does not separately fetch original images or other measurement attachments.

The complete snapshot always remains in `records.json`. When a filter is set,
`filtered_records.json` contains the matching full device records. Rerunning this
cell replaces only that filtered file. Copy the printed path into `INPUT_JSON`
in the CSV notebook to convert the filtered subset.

In [ ]:
records = load_records(records_file)
counts = Counter(kind for record in records for kind in measurement_types(record))
print(f"Total devices: {len(records)}")
print("Available measurement types:", dict(sorted(counts.items())))

selected = filter_records(records, MEASUREMENT)
selected_file = records_file
if MEASUREMENT is not None:
    selected_file = write_records(selected, records_file.parent / "filtered_records.json")
print(f"Selected devices: {len(selected)}")
print(f"CSV input: {selected_file}")

## Next step

Open [02_data_csv.ipynb](02_data_csv.ipynb). Its default input is the latest complete
download. Set an explicit file path there for a filtered subset or an older snapshot.